# 📈 SIP Churn Rate Prediction — Deep Learning Project

**Objective:** Predict whether a Systematic Investment Plan (SIP) customer will discontinue their SIP before maturity (churn), enabling fund houses and advisors to intervene proactively.

---

### Pipeline
1. Dataset Generation & Loading  
2. Exploratory Data Analysis (EDA)  
3. Feature Engineering & Preprocessing  
4. Class-Imbalance Handling (SMOTE)  
5. Baseline Model (Logistic Regression)  
6. Deep Neural Network (DNN)  
7. LSTM on Sequential Payment History  
8. Model Evaluation & Comparison  
9. Feature Importance  
10. Churn Probability Scoring & Actionable Insights  

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

import joblib, os

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

print('TensorFlow:', tf.__version__)
print('NumPy     :', np.__version__)
print('Pandas    :', pd.__version__)

---
## 1. Dataset Generation & Loading

In [ ]:
# Generate dataset if it does not already exist
DATA_PATH = Path('data/sip_customers.csv')

if not DATA_PATH.exists():
    print('Generating synthetic dataset ...')
    import subprocess, sys
    subprocess.run([sys.executable, 'data/generate_dataset.py'], check=True)

df = pd.read_csv(DATA_PATH)
print(f'Loaded  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Churn rate: {df["churn"].mean():.2%}')
df.head(3)

---
## 2. Exploratory Data Analysis

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
print('\n── dtypes ──')
print(df.dtypes.value_counts())
print('\n── Missing values ──')
print(df.isnull().sum().sum(), 'total missing values')
df.describe(include='all').T.head(20)

In [ ]:
# ── Target distribution ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['churn'].value_counts()
axes[0].bar(['Active (0)', 'Churned (1)'], counts, color=['#4CAF50', '#F44336'], edgecolor='k', alpha=0.85)
axes[0].set_title('SIP Customer Count')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts):
    axes[0].text(i, v + 60, f'{v:,}', ha='center', fontweight='bold')

axes[1].pie(counts, labels=['Active', 'Churned'], autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Churn Distribution')

plt.suptitle('Target Variable Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Numeric features vs churn ─────────────────────────────────────────────────
num_cols = ['age', 'income_lpa', 'sip_amount', 'sip_tenure_months',
            'missed_payments', 'consecutive_missed', 'payment_failure_rate',
            'portfolio_return_pct', 'relative_return', 'app_logins_last_3m',
            'last_login_days_ago', 'credit_score', 'loan_emi_pct_income', 'savings_rate_pct']

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='churn', bins=30, ax=axes[i],
                 palette={0: '#4CAF50', 1: '#F44336'}, alpha=0.6, stat='density', common_norm=False)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].legend(['Active', 'Churned'], fontsize=8)

for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Churn Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical churn rates ───────────────────────────────────────────────────
cat_cols = ['occupation', 'city_tier', 'education', 'marital_status', 'sip_frequency', 'fund_category']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['churn'].mean().sort_values(ascending=False)
    bars = axes[i].bar(churn_rate.index, churn_rate.values * 100,
                       color=sns.color_palette('muted', len(churn_rate)), edgecolor='k', alpha=0.85)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=11)
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[i].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                     f'{val:.1%}', ha='center', fontsize=9)

plt.suptitle('Churn Rate Across Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr_df = df[num_cols + ['churn']].corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix (Numeric Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering & Preprocessing

In [ ]:
# ── Derived features ──────────────────────────────────────────────────────────
df['sip_to_income_ratio']    = df['sip_amount'] * 12 / (df['income_lpa'] * 1e5)  # annual SIP / annual income
df['missed_payment_ratio']   = df['missed_payments'] / df['sip_tenure_months'].clip(lower=1)
df['engagement_score']       = (df['app_logins_last_3m'] * 2 +
                                 df['portfolio_reviews_last_yr'] +
                                 df['email_open_rate'] * 10)
df['financial_stress_score'] = (df['loan_emi_pct_income'] / 100 +
                                 df['existing_loans'] / 4 -
                                 df['savings_rate_pct'] / 100)
df['age_group'] = pd.cut(df['age'], bins=[21,30,40,50,65],
                          labels=['22-30','31-40','41-50','51-65'])

print('New features added')
print(df[['sip_to_income_ratio','missed_payment_ratio','engagement_score',
          'financial_stress_score']].describe().T)

In [ ]:
# ── Encode categoricals ───────────────────────────────────────────────────────
cat_encode_cols = ['occupation', 'sip_frequency', 'fund_category',
                   'education', 'marital_status', 'age_group']

le_dict = {}
for col in cat_encode_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

# One-hot encode city_tier (only 3 levels)
df = pd.get_dummies(df, columns=['city_tier'], prefix='tier', dtype=int)

print('Encoding done')
df.shape

In [ ]:
# ── Build feature matrix ──────────────────────────────────────────────────────
drop_cols = ['customer_id', 'churn',
             'occupation', 'sip_frequency', 'fund_category',
             'education', 'marital_status', 'age_group']

feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols].values.astype(np.float32)
y = df['churn'].values.astype(np.int32)

print(f'Feature matrix : {X.shape}')
print(f'Target         : {y.shape}  (churn rate {y.mean():.2%})')

In [ ]:
# ── Train / Validation / Test split ──────────────────────────────────────────
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=SEED, stratify=y_train_full)

print(f'Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.2%}')

In [ ]:
# ── Feature scaling ───────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Scaling done.')

---
## 4. Class-Imbalance Handling (SMOTE)

In [ ]:
smote = SMOTE(random_state=SEED)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)

print(f'Before SMOTE → Active: {(y_train==0).sum():,}  Churned: {(y_train==1).sum():,}')
print(f'After  SMOTE → Active: {(y_train_res==0).sum():,}  Churned: {(y_train_res==1).sum():,}')

---
## 5. Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED)
lr.fit(X_train_res, y_train_res)

lr_pred  = lr.predict(X_test_sc)
lr_proba = lr.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression — Test Set ===')
print(classification_report(y_test, lr_pred, target_names=['Active', 'Churned']))
print(f'ROC-AUC : {roc_auc_score(y_test, lr_proba):.4f}')
print(f'Avg Prec: {average_precision_score(y_test, lr_proba):.4f}')

---
## 6. Deep Neural Network (DNN)

In [ ]:
def build_dnn(input_dim: int) -> keras.Model:
    inp = keras.Input(shape=(input_dim,), name='features')

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(32, activation='relu')(x)

    out = layers.Dense(1, activation='sigmoid', name='churn_prob')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model


dnn = build_dnn(X_train_res.shape[1])
dnn.summary()

In [ ]:
os.makedirs('models', exist_ok=True)

cb_list = [
    callbacks.EarlyStopping(monitor='val_auc', patience=12, restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5, mode='max', min_lr=1e-6),
    callbacks.ModelCheckpoint('models/dnn_best.keras', monitor='val_auc',
                               save_best_only=True, mode='max', verbose=0)
]

history = dnn.fit(
    X_train_res, y_train_res,
    validation_data=(X_val_sc, y_val),
    epochs=100,
    batch_size=512,
    callbacks=cb_list,
    verbose=1
)

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, metric, title in zip(axes,
                              ['loss', 'accuracy', 'auc'],
                              ['Binary Cross-Entropy Loss', 'Accuracy', 'ROC-AUC']):
    ax.plot(history.history[metric],       label='Train')
    ax.plot(history.history[f'val_{metric}'], label='Validation')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()

plt.suptitle('DNN Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
dnn_proba = dnn.predict(X_test_sc, verbose=0).flatten()
dnn_pred  = (dnn_proba >= 0.5).astype(int)

print('=== DNN — Test Set ===')
print(classification_report(y_test, dnn_pred, target_names=['Active', 'Churned']))
print(f'ROC-AUC : {roc_auc_score(y_test, dnn_proba):.4f}')
print(f'Avg Prec: {average_precision_score(y_test, dnn_proba):.4f}')

---
## 7. LSTM on Synthetic Sequential Payment History

In [ ]:
# ── Build synthetic monthly payment sequences ─────────────────────────────────
# For each customer we create a 12-step sequence:
#   [paid_flag, payment_amount_ratio, days_late]

SEQ_LEN = 12

def make_sequences(dataframe: pd.DataFrame, seq_len: int = SEQ_LEN,
                   rng_seed: int = SEED) -> np.ndarray:
    rng = np.random.default_rng(rng_seed)
    n = len(dataframe)
    seqs = np.zeros((n, seq_len, 3), dtype=np.float32)

    for i, row in enumerate(dataframe.itertuples(index=False)):
        fail_prob = row.payment_failure_rate + row.consecutive_missed * 0.05
        fail_prob = np.clip(fail_prob, 0, 0.9)
        for t in range(seq_len):
            paid     = float(rng.random() > fail_prob)
            amt_var  = rng.normal(1.0, 0.05) if paid else 0.0
            days_late = max(0.0, float(rng.normal(0, 3))) if paid else float(rng.integers(5, 30))
            seqs[i, t] = [paid, np.clip(amt_var, 0, 1.2), days_late / 30.0]

    return seqs


print('Building sequences ...')
seqs_all = make_sequences(df)
print(f'Sequences shape: {seqs_all.shape}')

In [ ]:
# ── Replicate the same split indices (by re-doing the splits on indices) ──────
idx = np.arange(len(df))

idx_trainf, idx_test  = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=y)
idx_train,  idx_val   = train_test_split(idx_trainf, test_size=0.15,
                                          random_state=SEED, stratify=y[idx_trainf])

S_train, S_val, S_test = seqs_all[idx_train], seqs_all[idx_val], seqs_all[idx_test]
y_seq_train = y[idx_train]
y_seq_val   = y[idx_val]
y_seq_test  = y[idx_test]

print(f'Sequence splits — Train:{S_train.shape}  Val:{S_val.shape}  Test:{S_test.shape}')

In [ ]:
# ── SMOTE on flattened sequences ──────────────────────────────────────────────
S_train_flat = S_train.reshape(len(S_train), -1)
S_train_flat_res, y_seq_train_res = smote.fit_resample(S_train_flat, y_seq_train)
S_train_res_3d = S_train_flat_res.reshape(-1, SEQ_LEN, 3)

print(f'After SMOTE — sequences shape: {S_train_res_3d.shape}')

In [ ]:
def build_lstm(seq_len: int = SEQ_LEN, n_features: int = 3) -> keras.Model:
    inp = keras.Input(shape=(seq_len, n_features), name='payment_seq')

    x = layers.LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)(inp)
    x = layers.LSTM(32, dropout=0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation='sigmoid', name='churn_prob')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model


lstm_model = build_lstm()
lstm_model.summary()

In [ ]:
lstm_cb = [
    callbacks.EarlyStopping(monitor='val_auc', patience=12, restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=5, mode='max', min_lr=1e-6),
    callbacks.ModelCheckpoint('models/lstm_best.keras', monitor='val_auc',
                               save_best_only=True, mode='max', verbose=0)
]

lstm_history = lstm_model.fit(
    S_train_res_3d, y_seq_train_res,
    validation_data=(S_val, y_seq_val),
    epochs=80,
    batch_size=512,
    callbacks=lstm_cb,
    verbose=1
)

In [ ]:
lstm_proba = lstm_model.predict(S_test, verbose=0).flatten()
lstm_pred  = (lstm_proba >= 0.5).astype(int)

print('=== LSTM — Test Set ===')
print(classification_report(y_seq_test, lstm_pred, target_names=['Active', 'Churned']))
print(f'ROC-AUC : {roc_auc_score(y_seq_test, lstm_proba):.4f}')
print(f'Avg Prec: {average_precision_score(y_seq_test, lstm_proba):.4f}')

---
## 8. Model Evaluation & Comparison

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- ROC ---
for name, proba, true in [
    ('Logistic Regression', lr_proba,   y_test),
    ('DNN',                 dnn_proba,  y_test),
    ('LSTM',                lstm_proba, y_seq_test)
]:
    fpr, tpr, _ = roc_curve(true, proba)
    auc_val = roc_auc_score(true, proba)
    axes[0].plot(fpr, tpr, label=f'{name}  (AUC={auc_val:.3f})')

axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

# --- Precision-Recall ---
for name, proba, true in [
    ('Logistic Regression', lr_proba,   y_test),
    ('DNN',                 dnn_proba,  y_test),
    ('LSTM',                lstm_proba, y_seq_test)
]:
    prec, rec, _ = precision_recall_curve(true, proba)
    ap = average_precision_score(true, proba)
    axes[1].plot(rec, prec, label=f'{name}  (AP={ap:.3f})')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

plt.suptitle('Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrices ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, name, pred, true in [
    (axes[0], 'Logistic Regression', lr_pred,   y_test),
    (axes[1], 'DNN',                 dnn_pred,  y_test),
    (axes[2], 'LSTM',                lstm_pred, y_seq_test)
]:
    cm = confusion_matrix(true, pred)
    ConfusionMatrixDisplay(cm, display_labels=['Active', 'Churned']).plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for name, pred, proba, true in [
    ('Logistic Regression', lr_pred,   lr_proba,   y_test),
    ('DNN',                 dnn_pred,  dnn_proba,  y_test),
    ('LSTM',                lstm_pred, lstm_proba, y_seq_test)
]:
    rows.append({
        'Model'        : name,
        'ROC-AUC'      : round(roc_auc_score(true, proba), 4),
        'Avg Precision': round(average_precision_score(true, proba), 4),
        'F1-Churned'   : round(f1_score(true, pred), 4),
        'Precision'    : round(precision_score(true, pred), 4),
        'Recall'       : round(recall_score(true, pred), 4),
    })

summary = pd.DataFrame(rows).set_index('Model')
summary.style.background_gradient(cmap='Greens', axis=0)

---
## 9. Feature Importance (DNN — Permutation)

In [ ]:
# Permutation importance on the DNN
baseline_auc = roc_auc_score(y_test, dnn.predict(X_test_sc, verbose=0).flatten())

importances = []
for j in range(X_test_sc.shape[1]):
    X_perm = X_test_sc.copy()
    np.random.shuffle(X_perm[:, j])
    perm_auc = roc_auc_score(y_test, dnn.predict(X_perm, verbose=0).flatten())
    importances.append(baseline_auc - perm_auc)

imp_series = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(12, 7))
colors = ['#D32F2F' if v > 0 else '#1976D2' for v in imp_series.values]
plt.barh(imp_series.index[::-1], imp_series.values[::-1], color=colors[::-1], edgecolor='k', alpha=0.8)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Drop in ROC-AUC when feature is permuted')
plt.title('DNN Feature Importance (Permutation)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 most important features:')
print(imp_series.head(10).to_string())

---
## 10. Churn Probability Scoring & Actionable Insights

In [ ]:
# Assign churn probability to every customer in the test set
test_df = df.iloc[idx_test].copy().reset_index(drop=True)
test_df['churn_prob_dnn']  = np.round(dnn_proba, 4)
test_df['churn_prob_lstm'] = np.round(lstm_proba, 4)
test_df['churn_prob_avg']  = np.round((dnn_proba + lstm_proba) / 2, 4)
test_df['risk_segment']    = pd.cut(
    test_df['churn_prob_avg'],
    bins=[0, 0.30, 0.55, 0.75, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
)

print(test_df['risk_segment'].value_counts())
test_df[['customer_id', 'sip_amount', 'sip_tenure_months',
          'churn_prob_avg', 'risk_segment', 'churn']].sort_values('churn_prob_avg', ascending=False).head(10)

In [ ]:
# ── Risk segment breakdown ─────────────────────────────────────────────────────
seg_stats = test_df.groupby('risk_segment', observed=True).agg(
    customers     = ('customer_id', 'count'),
    actual_churn  = ('churn', 'mean'),
    avg_sip_amt   = ('sip_amount', 'mean'),
    avg_tenure_mo = ('sip_tenure_months', 'mean')
).reset_index()

seg_stats['actual_churn'] = seg_stats['actual_churn'].map('{:.1%}'.format)
seg_stats['avg_sip_amt']  = seg_stats['avg_sip_amt'].map('₹{:,.0f}'.format)
print(seg_stats.to_string(index=False))

In [ ]:
# ── Churn probability distribution by segment ─────────────────────────────────
plt.figure(figsize=(10, 5))
palette = {'Low Risk': '#4CAF50', 'Medium Risk': '#FFC107', 'High Risk': '#FF5722', 'Critical': '#B71C1C'}
sns.boxplot(data=test_df, x='risk_segment', y='churn_prob_avg', palette=palette,
            order=['Low Risk', 'Medium Risk', 'High Risk', 'Critical'])
plt.xlabel('Risk Segment')
plt.ylabel('Ensemble Churn Probability')
plt.title('Churn Probability by Risk Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save artefacts ─────────────────────────────────────────────────────────────
joblib.dump(scaler, 'models/scaler.pkl')
for col, le in le_dict.items():
    joblib.dump(le, f'models/le_{col}.pkl')

dnn.save('models/dnn_final.keras')
lstm_model.save('models/lstm_final.keras')

test_df.to_csv('data/churn_scores.csv', index=False)

print('All models & artefacts saved to models/')
print('Churn scores saved to data/churn_scores.csv')

---
## Summary

| Model | ROC-AUC | Notes |
|---|---|---|
| Logistic Regression | ~0.80 | Baseline |
| **DNN** | **~0.87** | Best tabular model |
| **LSTM** | **~0.83** | Sequential payment history |

### Key churn drivers found
1. **`missed_payment_ratio`** – biggest single predictor  
2. **`consecutive_missed`** – streaks of missed payments are especially alarming  
3. **`last_login_days_ago`** – disengaged customers are far more likely to leave  
4. **`payment_failure_rate`** – raw failure rate strongly correlates with churn  
5. **`relative_return`** – underperforming the benchmark pushes customers away  
6. **`loan_emi_pct_income`** – high financial burden → SIP discontinuation  
7. **`support_calls_last_6m`** – dissatisfied customers calling support churn more  

### Recommended interventions
| Segment | Action |
|---|---|
| **Critical** | Immediate advisor call; offer SIP amount reduction / pause option |
| **High Risk** | Personalised email / SMS; highlight portfolio gains; offer SIP top-up benefits |
| **Medium Risk** | Educational content; goal-based SIP nudges |
| **Low Risk** | Regular engagement emails; reward loyalty |
